# Model B
---

Load the preprocessed datasets, vocabulary, config and dataloaders using the following cell.

In [1]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
vocab_reversed = load_vocab(rev=True)
config = load_config()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [2]:
train_set.head()

,input,response
0,<S0> how do i navigate without a mouse?,"<S1> depends on the desktop, for kde it's ctrl..."
1,<S0> hey rafa <SEP> somebody is using the karm...,"<S1> karmic discussion in #ubuntu+1, not here,..."
2,<S0> is easybuntu still favored over automatix?,<S1> always will be
3,<S0> haha mate it's shockingly back <SEP> does...,"<S1> fairly often on a shell, though i'm on ko..."
4,<S0> what time of the day <PATH>/ is executed?,<S1> that is set in <PATH> i believe


In [3]:
config

{'MAX_LENGTH': 15,
 'MIN_TOKEN_FREQ': 10,
 'MIN_LENGTH': 4,
 'BATCH_SIZE': 64,
 'VOCAB_SIZE': 3023,
 'PAD_IDX': 0,
 'SOS_IDX': 1,
 'EOS_IDX': 2,
 'UNK_IDX': 3,
 'SEP_IDX': 4,
 'S0_IDX': 5,
 'S1_IDX': 6,
 'URL_IDX': 7,
 'IP_IDX': 8,
 'PATH_IDX': 9}

In [4]:
import torch

# Always use cuda if available (NVIDIA GPU), then try for apple silicon, otherwise just a plain old CPU (but training will be vastly slower)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'Using device: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print('Using device: MPS (Apple Silicon)')
else:
    device = torch.device("cpu")
    print('Using device: CPU')

Using device: MPS (Apple Silicon)


Quick plan:

1. Define the Encoder  — GRU that reads input, returns all hidden states + final hidden state (With attension)
2. Define the Decoder  — GRU that generates output token by token by looking back at all encoder states and compute a weighted average 
                         of the most relevant parts of the input sentence before predicting the next word. (With attension)
4. Define the Seq2Seq  — wrapper that connects encoder and decoder, handles teacher forcing
5. Training loop       — with validation loss monitoring
6. Inference function  — greedy decoding for generating responses
7. Evaluation          — BLEU score on test set + manual examples

## Encoder

---

The encoder will read the input sequences we've prepared, and will output a single final hidden state to initialise the decoder. So, the architecture is actually quite simple:
- An embedding layer (dataset has low GloVe coverage, so we can train embeddings from scratch).
- A stack of Gated Recurrent Units (combats vanishing gradients, fewer parameters to train than an LSTM, with comparable performance).
- Dropout as a regularisation technique to combat overfitting and improve generalisation (only applied during training). We'll apply dropout between the embedding layer, and between the layers in the GRU.

Defining a sequential model in PyTorch follows the usual pattern:
- Extend the [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) base class.
- Initialise your layers in the constructor.
- Override the `forward` method, explicitly feeding X inputs through your layers.

Please read the [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) documentation, it explains a lot! :)

In [5]:
from encoder import Encoder

## Decoder

---

The decoder must first initialise its hidden state with the final hidden state of the encoder. Then, at each timestep (a forward pass through the network processing one token):

1- Obtain the embedding for the previous token

2- Pass it through the GRU

3- Compute attention scores between the decoder output and the encoder outputs

4- Use the attention weights to compute a context vector as a weighted sum of encoder hidden states

5- Combine the decoder output and the context vector

6- Project the combined representation to the vocabulary size to predict the next token (using a fully connected linear layer)

Outside of this, its layer initialisation is very similar to the encoder.

Note that we also train separate embeddings in the decoder rather than using shared embeddings between the encoder and decoder (called weight tying). This keeps the embeddings specialised, as the encoder and decoder objectives differ: the encoder must encode the meaning of the input sequence into hidden representations, while the decoder must use these representations to predict the next token given the previous one. Allowing each component to learn token representations optimised for its objective improves modelling flexibility.

In [6]:
from decoder import Decoder

## Seq2Seq Wrapper

---

This is the wrapper model that will be built from the encoder and decoder. It will:
1. Pass the full input sequence through the encoder to produce a final hidden state
2. Use this final hidden state to initialise the decoder
3. Loop through each decoding timestep and feed each token in one at a time
4. Apply **teacher forcing** during training only (feeding the ground truth labels in at each timestep to improve stability and reduce _exposure bias_I)

The variables passed into the `forward` method are those we prepared at the data preparation stage:
- `encoder_input`: the encoded input sequence (for the encoder)
- `decoder_input`: the encoded response with SOS prepended

In [7]:
from seq2Seq import Seq2Seq

## Training Loop

---

This is the main training loop, where we'll use cross entropy loss, Adam optimiser, and gradient clipping to manage exploding gradients (this is common in BPTT).

We can use a `max_norm=1.0` to scale the gradient vector down if its norm exceeds 1 to help keep the weight updates stable.

First things first: let's define the training and model hyperparameters, and instantiate the model.

In [9]:
# Training Parameters
EPOCHS = 80
CLIP_MAX_NORM = 50.0
TEACHER_FORCING_PROBA = 0.5

# Model Hyperparameters
ENCODER_EMBED_DIM = 128
ENCODER_HIDDEN_DIM = 256
ENCODER_NUM_LAYERS = 2
ENCODER_DROPOUT_PROBA = 0.3

DECODER_EMBED_DIM = 128
DECODER_HIDDEN_DIM = 256
DECODER_NUM_LAYERS = 2
DECODER_DROPOUT_PROBA = 0.3

ENCODER_LEARNING_RATE=1e-3
DECODER_LEARNING_RATE=ENCODER_LEARNING_RATE * 2

model_b = Seq2Seq(
    encoder=Encoder(
        vocab_size=config['VOCAB_SIZE'], 
        embed_dim=ENCODER_EMBED_DIM, 
        padding_idx=config['PAD_IDX'], 
        hidden_dim=ENCODER_HIDDEN_DIM,
        num_layers=ENCODER_NUM_LAYERS,
        dropout_proba=ENCODER_DROPOUT_PROBA,
        use_attention=True
    ),
    decoder = Decoder(
        vocab_size=config['VOCAB_SIZE'],
        embed_dim=DECODER_EMBED_DIM,
        padding_idx=config['PAD_IDX'],
        hidden_dim=DECODER_HIDDEN_DIM,
        num_layers=DECODER_NUM_LAYERS,
        dropout_proba=DECODER_DROPOUT_PROBA,
        use_attention=True
    ),
    device=device,
    use_attention=True
).to(device) # Move model to device

In [10]:
#!mkdir -p models/model_b
from pathlib import Path

Path("models/model_b").mkdir(parents=True, exist_ok=True)

In [ ]:
from train import train

train(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    checkpoint_dir='models/model_b',
    hyperparams={
        'EPOCHS': EPOCHS,
        'CLIP_MAX_NORM': CLIP_MAX_NORM,
        'TEACHER_FORCING_PROBA': TEACHER_FORCING_PROBA,
        'ENCODER_LEARNING_RATE': ENCODER_LEARNING_RATE,
        'DECODER_LEARNING_RATE': DECODER_LEARNING_RATE
    }
)

No checkpoints, training model from scratch...
  [5%] Batch 20/410 | Avg Loss: 7.8592
  [10%] Batch 40/410 | Avg Loss: 7.3582
  [15%] Batch 60/410 | Avg Loss: 6.9221
  [20%] Batch 80/410 | Avg Loss: 6.6223
  [24%] Batch 100/410 | Avg Loss: 6.4279
  [29%] Batch 120/410 | Avg Loss: 6.2993
  [34%] Batch 140/410 | Avg Loss: 6.1979
  [39%] Batch 160/410 | Avg Loss: 6.1246
  [44%] Batch 180/410 | Avg Loss: 6.0629
  [49%] Batch 200/410 | Avg Loss: 6.0132
  [54%] Batch 220/410 | Avg Loss: 5.9748
  [59%] Batch 240/410 | Avg Loss: 5.9417
  [63%] Batch 260/410 | Avg Loss: 5.9130
  [68%] Batch 280/410 | Avg Loss: 5.8865
  [73%] Batch 300/410 | Avg Loss: 5.8649
  [78%] Batch 320/410 | Avg Loss: 5.8430
  [83%] Batch 340/410 | Avg Loss: 5.8261
  [88%] Batch 360/410 | Avg Loss: 5.8076
  [93%] Batch 380/410 | Avg Loss: 5.7945
  [98%] Batch 400/410 | Avg Loss: 5.7820
Computing BLEU for epoch...
  BLEU: batch 1/10...
  BLEU: batch 4/10...
  BLEU: batch 7/10...
  BLEU: batch 10/10...
Epoch 1/10 | Train Lo